# Online Retail Data Cleaning

This notebook performs data loading, inspection, cleaning, and preprocessing
of the Online Retail dataset.

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully")

Libraries imported successfully


In [40]:
import pandas as pd

df = pd.read_excel(
    file_path,
    sheet_name="Online Retail",
    engine="openpyxl"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 2. Initial Dataset Inspection

Before cleaning the data, we inspect its structure, columns, data types,
missing values, duplicate records, and basic statistical information.

In [41]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 541909
Number of columns: 8


In [42]:
print("Columns:")
for column in df.columns:
    print("-", column)

Columns:
- InvoiceNo
- StockCode
- Description
- Quantity
- InvoiceDate
- UnitPrice
- CustomerID
- Country


In [43]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [45]:
missing_values = df.isnull().sum()

print("Missing values:")
print(missing_values)

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [46]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 5268


In [47]:
print("Unique values in each column:\n")

for column in df.columns:
    print(f"{column}: {df[column].nunique()}")

Unique values in each column:

InvoiceNo: 25900
StockCode: 4070
Description: 4223
Quantity: 722
InvoiceDate: 23260
UnitPrice: 1630
CustomerID: 4372
Country: 38


In [48]:
print(df.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


## 3. Data Cleaning

The dataset will be cleaned by:
- Removing duplicate transactions
- Handling missing values
- Correcting data types
- Removing invalid quantities and prices
- Creating useful transaction-level features

In [49]:
clean_df = df.copy()

print("Original shape:", df.shape)
print("Working dataset shape:", clean_df.shape)

Original shape: (541909, 8)
Working dataset shape: (541909, 8)


In [50]:
duplicates = clean_df.duplicated().sum()

print("Duplicate rows found:", duplicates)

clean_df = clean_df.drop_duplicates()

print("Shape after removing duplicates:", clean_df.shape)

Duplicate rows found: 5268
Shape after removing duplicates: (536641, 8)


In [51]:
print(clean_df.isnull().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64


In [52]:
clean_df = clean_df.dropna(subset=["CustomerID"])

print("Shape after removing missing CustomerID:")
print(clean_df.shape)

Shape after removing missing CustomerID:
(401604, 8)


In [53]:
clean_df = clean_df.dropna(subset=["Description"])

print("Shape after removing missing Description:")
print(clean_df.shape)

Shape after removing missing Description:
(401604, 8)


In [54]:
clean_df["InvoiceDate"] = pd.to_datetime(clean_df["InvoiceDate"])

clean_df["CustomerID"] = clean_df["CustomerID"].astype(int)

print(clean_df.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              int64
Country                   str
dtype: object


In [55]:
print("Quantity <= 0:", (clean_df["Quantity"] <= 0).sum())

Quantity <= 0: 8872


In [56]:
clean_df = clean_df[clean_df["Quantity"] > 0]

print("Shape after quantity cleaning:", clean_df.shape)

Shape after quantity cleaning: (392732, 8)


In [57]:
print("UnitPrice <= 0:", (clean_df["UnitPrice"] <= 0).sum())

UnitPrice <= 0: 40


In [58]:
clean_df = clean_df[clean_df["UnitPrice"] > 0]

print("Shape after price cleaning:", clean_df.shape)

Shape after price cleaning: (392692, 8)


In [59]:
clean_df["TotalAmount"] = (
    clean_df["Quantity"] * clean_df["UnitPrice"]
)

clean_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [60]:
print("Final dataset shape:", clean_df.shape)

print("\nMissing values:")
print(clean_df.isnull().sum())

print("\nDuplicate rows:")
print(clean_df.duplicated().sum())

Final dataset shape: (392692, 9)

Missing values:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalAmount    0
dtype: int64

Duplicate rows:
0


In [61]:
clean_df.to_csv(
    "../data/processed/online_retail_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
